# Web scraping

<div style="display:flex;justify-content:space-between;align-items:center;background:#0d0d0f;border:1px solid #1e1e24;border-left:4px solid #4fc3f7;border-radius:6px;padding:clamp(1rem,2.5vw,1.8rem) clamp(1.2rem,3vw,2.4rem);margin-bottom:2rem;position:relative;overflow:hidden;box-shadow:0 4px 32px rgba(0,0,0,0.5);font-family:'Segoe UI',sans-serif;">
<div style="position:absolute;top:0;left:0;right:0;bottom:0;background:radial-gradient(ellipse at 0% 50%,rgba(79,195,247,0.07) 0%,transparent 60%);pointer-events:none;"></div>
<div style="display:flex;flex-direction:column;gap:0.3rem;">
<p style="font-size:clamp(1.3rem,3.5vw,2.4rem);color:#f0f0f5;margin:0;line-height:1.1;font-weight:700;letter-spacing:-0.01em;">Web-Scraping mit BeautifulSoup</p>
<p style="font-size:clamp(0.75rem,1.6vw,1rem);color:#7a7a90;margin:0;letter-spacing:0.04em;font-weight:300;">Development Expert Python / PCAP &nbsp;|&nbsp; Kapitel 12: Automatisierungen &nbsp;|&nbsp; Notebook 12b</p>
</div>
</div>

**Legende**

> **[Kursinhalt]** Dieses Notebook ist kein PCAP-Pruefungsinhalt

---

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#4fc3f7;background:rgba(79,195,247,0.12);border:1px solid rgba(79,195,247,0.3);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
1. Warum Scraping? Wenn es keine API gibt
</span>
</div>

Im letzten Notebook haben wir gelernt wie man Daten ueber eine REST-API abruft -- strukturiert, mit JSON, genau fuer Programme gemacht.

Aber nicht jede Website hat eine API. Manchmal liegen die Daten die man braucht einfach auf einer normalen Webseite -- als HTML, fuer Menschen lesbar. Preislisten, Stellenanzeigen, Sportergebnisse, Buchtitel -- riesige Mengen an Informationen die keine maschinenlesbare Schnittstelle haben.

**Web-Scraping** ist das automatisierte Abrufen und Extrahieren von Daten aus HTML-Seiten. Das Prinzip:

1. Seite mit `requests` herunterladen -- das HTML als Text
2. HTML mit `BeautifulSoup` parsen -- eine navigierbare Struktur aufbauen
3. Die gewuenschten Daten aus der Struktur herausziehen

Bevor wir das tun, muessen wir verstehen was HTML eigentlich ist.

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#4fc3f7;background:rgba(79,195,247,0.12);border:1px solid rgba(79,195,247,0.3);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
2. HTML -- die Sprache des Webs
</span>
</div>

HTML (HyperText Markup Language) beschreibt die Struktur einer Webseite. Es besteht aus **Tags** -- Elementen in spitzen Klammern die Text umschliessen:

```html
<h1>Das ist eine Ueberschrift</h1>
<p>Das ist ein Absatz.</p>
<a href="https://example.com">Das ist ein Link</a>
```

Jedes Element kann **Attribute** haben -- Zusatzinformationen im oeffnenden Tag:
- `href` bei Links -- die Zieladresse
- `class` -- CSS-Klasse fuer Styling
- `id` -- eindeutiger Bezeichner

Elemente koennen verschachtelt werden und bilden so eine **Baumstruktur**:

```html
<div class="held">
    <h2>Aldric</h2>
    <p class="klasse">Krieger</p>
    <span class="hp">100</span>
</div>
```

BeautifulSoup macht diese Baumstruktur navigierbar -- man kann nach Tags, Klassen und IDs suchen.

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#4fc3f7;background:rgba(79,195,247,0.12);border:1px solid rgba(79,195,247,0.3);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
3. BeautifulSoup installieren und HTML parsen
</span>
</div>

In [ ]:
# Installation (einmalig)
# pip install beautifulsoup4 requests

try:
    from bs4 import BeautifulSoup
    import requests
    print('Alles installiert')
except ImportError as e:
    print(f'Fehlt: {e}')
    print('Bitte ausfuehren: pip install beautifulsoup4 requests')

In [ ]:
from bs4 import BeautifulSoup

# HTML-String direkt parsen -- zum Lernen des Prinzips
html = """
<html>
  <body>
    <h1>Heldenliste</h1>
    <div class="held">
      <h2>Aldric</h2>
      <p class="klasse">Krieger</p>
      <span class="hp">100</span>
    </div>
    <div class="held">
      <h2>Lyria</h2>
      <p class="klasse">Magierin</p>
      <span class="hp">80</span>
    </div>
  </body>
</html>
"""

# HTML parsen -- 'html.parser' ist der eingebaute Parser
soup = BeautifulSoup(html, 'html.parser')

print(type(soup))           # <class 'bs4.BeautifulSoup'>
print(soup.title)           # None -- kein <title> vorhanden
print(soup.h1)              # <h1>Heldenliste</h1>
print(soup.h1.text)         # 'Heldenliste'  -- nur der Text

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#4fc3f7;background:rgba(79,195,247,0.12);border:1px solid rgba(79,195,247,0.3);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
4. find() und find_all() -- Elemente suchen
</span>
</div>

Die zwei wichtigsten Methoden:

- `find(tag)` -- erstes passendes Element, oder `None` wenn nicht gefunden
- `find_all(tag)` -- alle passenden Elemente als Liste

Beide koennen nach Tag-Namen, CSS-Klassen, IDs und Attributen suchen.

In [ ]:
from bs4 import BeautifulSoup

html = """
<html><body>
  <h1>Heldenliste</h1>
  <div class="held">
    <h2>Aldric</h2>
    <p class="klasse">Krieger</p>
    <span class="hp">100</span>
  </div>
  <div class="held">
    <h2>Lyria</h2>
    <p class="klasse">Magierin</p>
    <span class="hp">80</span>
  </div>
  <div class="npc">
    <h2>Haendler</h2>
  </div>
</body></html>
"""
soup = BeautifulSoup(html, 'html.parser')

# find() -- erstes Element
erster_held = soup.find('div', class_='held')   # class_ wegen Python-Konflikt
print(erster_held.find('h2').text)   # Aldric

# find_all() -- alle Elemente
alle_helden = soup.find_all('div', class_='held')
print(f'{len(alle_helden)} Helden gefunden')   # 2 -- nicht 3, NPC hat andere Klasse

# Alle h2-Tags
alle_namen = soup.find_all('h2')
print([tag.text for tag in alle_namen])   # ['Aldric', 'Lyria', 'Haendler']

In [ ]:
# Strukturiert durch Elemente navigieren
alle_helden = soup.find_all('div', class_='held')

for held in alle_helden:
    name   = held.find('h2').text
    klasse = held.find('p', class_='klasse').text
    hp     = int(held.find('span', class_='hp').text)
    print(f'{name} ({klasse}) -- {hp} HP')

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#4fc3f7;background:rgba(79,195,247,0.12);border:1px solid rgba(79,195,247,0.3);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
5. Attribute auslesen -- Links und Bilder extrahieren
</span>
</div>

Manchmal will man nicht den Text eines Elements, sondern ein Attribut -- z.B. das `href` eines Links oder das `src` eines Bildes. Attribute werden wie Dictionary-Eintraege abgerufen.

In [ ]:
from bs4 import BeautifulSoup

html = """
<div>
  <a href="https://python.org">Python</a>
  <a href="https://github.com">GitHub</a>
  <a href="/relativ/pfad">Relativ</a>
  <img src="bild.png" alt="Ein Bild">
</div>
"""
soup = BeautifulSoup(html, 'html.parser')

# Alle Links extrahieren
alle_links = soup.find_all('a')
for link in alle_links:
    text = link.text
    url  = link['href']    # Attribut wie Dictionary
    print(f'{text}: {url}')

print()

# Nur externe Links (beginnen mit http)
externe_links = [a['href'] for a in soup.find_all('a')
                 if a['href'].startswith('http')]
print('Externe Links:', externe_links)

# Bild-URL
bild = soup.find('img')
print('Bild-src:', bild['src'])
print('Alt-Text:', bild.get('alt', 'kein Alt-Text'))  # .get() sicherer

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#4fc3f7;background:rgba(79,195,247,0.12);border:1px solid rgba(79,195,247,0.3);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
6. Eine echte Seite scrapen
</span>
</div>

Jetzt kombinieren wir `requests` und `BeautifulSoup`. Wir benutzen `books.toscrape.com` -- eine Seite die genau fuer Scraping-Uebungen gebaut wurde, kein echtes Buchgeschaeft.

In [ ]:
import requests
from bs4 import BeautifulSoup

URL = 'https://books.toscrape.com'

# Schritt 1: Seite herunterladen
response = requests.get(URL)
print(f'Statuscode: {response.status_code}')
print(f'Encoding:   {response.encoding}')
print(f'HTML-Laenge: {len(response.text)} Zeichen')

In [ ]:
import requests
from bs4 import BeautifulSoup

response = requests.get('https://books.toscrape.com')
soup     = BeautifulSoup(response.text, 'html.parser')

# Schritt 2: Struktur erkunden -- im Browser: Rechtsklick -> Untersuchen
# Jedes Buch steckt in einem <article class="product_pod">
buecher = soup.find_all('article', class_='product_pod')
print(f'{len(buecher)} Buecher auf der Seite')

# Schritt 3: Daten extrahieren
for buch in buecher[:5]:   # erste 5
    titel = buch.find('h3').find('a')['title']   # title-Attribut hat vollen Namen
    preis = buch.find('p', class_='price_color').text
    print(f'{titel[:40]:40} {preis}')

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#4fc3f7;background:rgba(79,195,247,0.12);border:1px solid rgba(79,195,247,0.3);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
7. Mehrere Seiten scrapen -- Paginierung
</span>
</div>

Die meisten Websites zeigen Daten auf mehreren Seiten. Um alle Daten zu bekommen, muss man automatisch durch die Seiten navigieren -- Paginierung.

In [ ]:
import requests
from bs4 import BeautifulSoup

BASE_URL = 'https://books.toscrape.com/catalogue'
alle_buecher = []
seite = 1
max_seiten = 3   # nur 3 Seiten fuer die Demo

while seite <= max_seiten:
    url = f'{BASE_URL}/page-{seite}.html'
    response = requests.get(url)

    if response.status_code != 200:
        print(f'Seite {seite} nicht erreichbar -- stoppe')
        break

    soup    = BeautifulSoup(response.text, 'html.parser')
    buecher = soup.find_all('article', class_='product_pod')

    for buch in buecher:
        titel = buch.find('h3').find('a')['title']
        preis = buch.find('p', class_='price_color').text.strip()
        # Sternebewertung aus class-Name extrahieren (One, Two, Three...)
        bewertung = buch.find('p', class_='star-rating')['class'][1]
        alle_buecher.append({'titel': titel, 'preis': preis, 'bewertung': bewertung})

    print(f'Seite {seite}: {len(buecher)} Buecher geladen')
    seite += 1

print(f'\nGesamt: {len(alle_buecher)} Buecher')
print('\nBeispiel:')
for b in alle_buecher[:3]:
    print(f'  {b["titel"][:40]:40} {b["preis"]:8}  Sterne: {b["bewertung"]}')

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#4fc3f7;background:rgba(79,195,247,0.12);border:1px solid rgba(79,195,247,0.3);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
8. Praktische Hinweise und Grenzen
</span>
</div>

**robots.txt -- was erlaubt ist:**

Jede Website kann unter `/robots.txt` festlegen welche Bereiche automatisiert abgerufen werden duerfen. Ein verantwortungsvoller Scraper prueft das vorher:

```
https://books.toscrape.com/robots.txt
```

**Hoeflicherkeit -- den Server nicht ueberlasten:**

Zu viele Requests in kurzer Zeit koennen einen Server ueberlasten -- und fuehren dazu, dass die eigene IP gesperrt wird. Eine kurze Pause zwischen Requests ist gute Praxis.

**Grenzen von BeautifulSoup:**

BeautifulSoup funktioniert nur mit statischem HTML -- dem HTML das der Server direkt liefert. Viele moderne Seiten laden Inhalte nachtraeglich per JavaScript. Fuer solche Seiten braucht man `Selenium` oder `Playwright` -- das sind Werkzeuge die einen echten Browser steuern.

In [ ]:
import requests
from bs4 import BeautifulSoup
import time

# Robustes Scraping-Muster mit Pausen und Fehlerbehandlung
def scrape_seite(url):
    try:
        # User-Agent setzen -- manche Server blockieren Standard-requests-Header
        headers = {'User-Agent': 'Mozilla/5.0 (Educational scraper)'}
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        return BeautifulSoup(response.text, 'html.parser')
    except requests.RequestException as e:
        print(f'Fehler bei {url}: {e}')
        return None


urls = [
    'https://books.toscrape.com/catalogue/page-1.html',
    'https://books.toscrape.com/catalogue/page-2.html',
]

for url in urls:
    soup = scrape_seite(url)
    if soup:
        buecher = soup.find_all('article', class_='product_pod')
        print(f'{url[-10:]}: {len(buecher)} Buecher')
    time.sleep(1)   # 1 Sekunde Pause -- hoeflich gegenueber dem Server

---

**Zusammenfassung**

| Konzept | Erklaerung |
|---------|------------|
| Web-Scraping | Automatisches Extrahieren von Daten aus HTML-Seiten |
| HTML-Tag | `<div class="x">Inhalt</div>` -- Name, Attribute, Inhalt |
| `BeautifulSoup(html, 'html.parser')` | HTML parsen -- navigierbare Struktur aufbauen |
| `soup.find(tag)` | Erstes passendes Element oder `None` |
| `soup.find_all(tag)` | Liste aller passenden Elemente |
| `class_='name'` | Nach CSS-Klasse suchen -- `class_` wegen Python-Konflikt |
| `element.text` | Nur den Textinhalt -- ohne HTML-Tags |
| `element['href']` | Attribut auslesen -- wie Dictionary |
| `element.get('attr', default)` | Attribut sicher auslesen |
| `robots.txt` | Prueft was erlaubt ist |
| `time.sleep(1)` | Pause zwischen Requests -- Server nicht ueberlasten |
| Grenze | BeautifulSoup funktioniert nicht bei JavaScript-geladenen Inhalten |